# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR⁲ dataset ["Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. We will step through data loading, inspection, extraction, exploratory analysis, and visualizations.

### Dataset Source
The dataset is referenced via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Croissant schema version: {metadata.conforms_to if hasattr(metadata,'conforms_to') else metadata.get('conformsTo','N/A')}")

## 2. Data Overview
Let's examine the available record sets, their corresponding `@id`s, and the fields within each record set. This aids in understanding the dataset structure and how data is organized.


In [ ]:
# List all available record sets and their fields by @id
print("Available record sets and fields:")

record_sets = dataset.record_sets  # This is a list of RecordSet objects
rs_ids = []
for rs in record_sets:
    print(f"- Record set name: {rs.name}, @id: {rs.id}")
    rs_ids.append(rs.id)
    print("  Fields (by @id and name):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()
# Store one record_set_id for use below
main_record_set_id = rs_ids[0] if rs_ids else None

## 3. Data Extraction
Load data from each record set into a DataFrame using their `@id`. This structure allows you to analyze any record set by simply switching to its `@id`.


In [ ]:
# Extract data from each record set by @id
dataframes = {}

for record_set_id in rs_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")

# Display columns in the main record set
print("\nColumns in the main record set DataFrame:")
if main_record_set_id:
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
We'll process one of the main numeric fields. For demonstration, select the first numeric field present in the main record set, filter by a threshold, normalize, and optionally group by a categorical attribute. All columns are referenced/by their `@id` as shown in the overview above.


In [ ]:
# Identify a numeric field in the main record set
main_rs = None
for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        main_rs = rs
        break

numeric_field_id = None
group_field_id = None
# Choose the first float or integer field
if main_rs is not None:
    for field in main_rs.fields:
        if field.data_type in ('schema:Float', 'schema:Integer', 'Float', 'Integer', 'Number', 'schema:Number'):
            numeric_field_id = field.id
            break
    # Optionally pick a categorical field for grouping
    for field in main_rs.fields:
        if numeric_field_id and field.data_type in ('schema:Text', 'Text', 'schema:Category', 'Category') and field.id != numeric_field_id:
            group_field_id = field.id
            break

# Use the id as the column name when referencing DataFrame columns, as provided by records()
df = dataframes[main_record_set_id]

if numeric_field_id and numeric_field_id in df.columns:
    # Convert to numeric (may be loaded as object if CSV had quotes etc)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = df[numeric_field_id].median()  # e.g. median as demonstration threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, normalized_col]].head())

    # Group if possible
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical group field found.")
else:
    print("No numeric field found in the selected record set.")

## 5. Visualization
Visualize the distribution of the numeric field and any grouping field if available. This gives intuition about data ranges and potential relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
We have demonstrated how to discover the contents of the FAIR⁲ mass spectrometry dataset through the Croissant schema, including programmatic field access by `@id`, filtering, normalization, grouping, and visualization. This workflow provides a reproducible foundation for robust data analysis and integration using standardized metadata.

- All record sets and fields are referenced by `@id` for clarity and reproducibility.
- You can further adapt this template for more advanced analytics or integration into ML pipelines.

**Next Steps:**
- Explore additional record sets by updating `main_record_set_id`.
- Apply domain-specific normalization or transformation to the protein data.
- Integrate outputs with downstream bioinformatics tools.